#  Day 1  AI Landscape & Your First Model
### AI Workshop · Google Colab Notebook

---

**Welcome.** This notebook is your hands-on companion for Day 1. You'll:
- Run a real AI model in under 10 lines of Python
- Swap models and observe how they behave differently
- Explore the Hugging Face model ecosystem
- Build your first intuition for the **Data → Model → Output → Integration** pipeline

**How to use this notebook:**
- Click on any code cell and press **Shift + Enter** to run it
- Run cells **in order** from top to bottom
- If you get an error, **read the last line**  it usually tells you exactly what went wrong
- The 🟡 cells are exercises for you. The 🔵 cells are demos  read and run them.

>  **New to Python?** That's fine. You don't need to write code today  you need to *run* code and *read* outputs. Focus on understanding what each cell *does*, not every line of syntax.

---
## Section 0  Setup
Run this first. It installs the libraries we need and pre-downloads the models.

> ⏱️ This takes about 60–90 seconds. Do it while the instructor is talking.

In [ ]:
# Install Hugging Face Transformers and Pillow (image library)
# This only needs to run once per session
!pip install transformers pillow requests -q

print(" Libraries installed.")

In [ ]:
# Pre-download models so we don't wait during exercises
# This runs silently  it's fetching model weights from Hugging Face
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

print("Downloading image classifier... (this is ~350MB, takes ~30s)")
image_classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

print("Downloading text classifier...")
sentiment_analyzer = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("\n All models ready. You're set for the session!")

---
## Section 1  The Core Mental Model

Before we run anything, let's understand the pipeline everything in AI follows:

```
DATA  ──→  MODEL  ──→  OUTPUT  ──→  INTEGRATION
```

| Step | What it means | Example |
|---|---|---|
| **Data** | The raw input you feed the model | A photo of a dog |
| **Model** | A trained neural network that processes input | ViT image classifier |
| **Output** | What the model produces | `{"label": "golden_retriever", "score": 0.94}` |
| **Integration** | How the output connects to something useful | Auto-tag the photo in your app |

**Everything you build in this workshop** is a version of this pipeline. Keep coming back to it.

---
## Section 2  🔵 Demo: Your First AI Model

We'll use **Google's ViT (Vision Transformer)**  a state-of-the-art image classification model trained on 14 million images across 1,000 categories.

Watch the instructor run this first, then you'll run it yourself.

In [ ]:
# ── HELPER FUNCTION ──────────────────────────────────────────────────────────
# This loads an image from a URL. You don't need to understand every line.
# Just know: give it a URL, it gives you an image object.

def load_image_from_url(url):
    """Load an image from a URL and return a PIL Image object."""
    response = requests.get(url, timeout=10)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    return img

print("Helper function ready.")

In [ ]:
# ── DEMO: IMAGE CLASSIFICATION ───────────────────────────────────────────────
# DATA: an image from the internet
# MODEL: Google ViT (already loaded in Section 0)
# OUTPUT: top predictions with confidence scores

# Load an image
image_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQR6A_mJG_LYa0hv3k_JyaNI2t6OIOrt_-xPQ&s"
image = load_image_from_url(image_url)

# Display it
print("Input image:")
display(image.resize((300, 300)))

# Run the model
print("\nRunning model...")
results = image_classifier(image, top_k=5)

# Show the output
print("\n📊 Model Output (Data → Model → OUTPUT):")
print("-" * 45)
for result in results:
    label = result['label'].replace('_', ' ').title()
    score = result['score']
    bar = '█' * int(score * 30)
    print(f"{label:<30} {bar} {score:.1%}")

**Read the output:**
- Each line is a *prediction* (label) with a *confidence score* (percentage)
- The model doesn't "see"  it compares pixel patterns to patterns it learned from 14 million training images
- Notice: it gives top-5 predictions, not just one. Real systems often use all of them.

**Q: Where is the Data → Model → Output pipeline here?**
- Data = the image URL
- Model = `image_classifier` (the ViT model)
- Output = the dictionary with `label` and `score`
- Integration = in a real app, you'd use this output to auto-tag photos

---
## Section 3  🟡 Exercise 1: Run It Yourself

**Your turn.** Change the image URL to something you choose and run the classifier.

**How to find image URLs:**
- Right-click any image on Google Images → "Copy image address"
- Use https://unsplash.com (free, high quality)
- Use any `.jpg` or `.png` URL

**Goal:** Get the model to:
1. Classify something correctly (high confidence)
2. Classify something *incorrectly* or with low confidence (try an unusual image)

In [ ]:
# 🟡 YOUR TURN  change the URL below to your chosen image
# ─────────────────────────────────────────────────────────

my_image_url = "PASTE YOUR IMAGE URL HERE"

# ─────────────────────────────────────────────────────────
# Don't change anything below this line

my_image = load_image_from_url(my_image_url)
display(my_image.resize((300, 300)))

my_results = image_classifier(my_image, top_k=5)

print("\n Results for your image:")
print("-" * 45)
for result in my_results:
    label = result['label'].replace('_', ' ').title()
    score = result['score']
    bar = '█' * int(score * 30)
    print(f"{label:<30} {bar} {score:.1%}")

# Reflection question (think about it, don't need to code the answer)
top_label = my_results[0]['label'].replace('_', ' ')
top_score = my_results[0]['score']
print(f"\n The model is {top_score:.0%} confident this is '{top_label}'.")
print("   Do you agree? What might explain any errors?")

**Reflect before moving on:**
- Was the model right? How confident was it?
- If it was wrong  why do you think that happened?
- What kind of images do you think this model was *not* trained on? (Hint: think about what was in ImageNet  mostly Western, everyday objects)

---
## Section 4  🟡 Exercise 2: Swap the Model

One of the most powerful things about Hugging Face is that **swapping models is one line of code**.

Below are 4 different model IDs  each does something different. Try at least 2 of them.

| Task | Model ID | What it does |
|---|---|---|
| Image classification | `google/vit-base-patch16-224` | Classifies 1000 ImageNet categories |
| Image classification (food) | `nateraw/food` | Classifies 101 types of food |
| Image classification (places) | `google/efficientnet-b7` | Broader category recognition |
| Emotion from image | `Rajaram1996/facial_emotion_vit` | Detects emotion in faces |

>  **Want to explore more?** Go to https://huggingface.co/models and filter by task = "Image Classification". There are thousands of models.

In [ ]:
# 🟡 SWAP THE MODEL
# Change the model_id below to one from the table above (or find your own on HF)
# ─────────────────────────────────────────────────────────────────────────────

model_id = "nateraw/food"   # ← change this

# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading model: {model_id}")
print("(This downloads if it's the first time  may take 30–60 seconds)\n")

new_classifier = pipeline("image-classification", model=model_id)

# Test it with your image from Exercise 1, or paste a new URL here
test_url = "https://external-preview.redd.it/10-legendary-regional-fast-food-chains-you-must-try-v0-3vuI5fbpKzyZLqOQ3PuHyQMxz990DMlU6JZiVuSCbH0.jpg?width=1080&crop=smart&auto=webp&s=edbe71e8be9cef48fcef730d9b8f7abc82851a5c"

test_image = load_image_from_url(test_url)
display(test_image.resize((300, 300)))

new_results = new_classifier(test_image, top_k=5)

print(f"\n📊 Results from '{model_id}':")
print("-" * 50)
for result in new_results:
    label = result['label'].replace('_', ' ').title()
    score = result['score']
    bar = '█' * int(score * 30)
    print(f"{label:<35} {bar} {score:.1%}")

print("\n How are these results different from the first model? Why?")

**Key insight:** The model ID is the only thing that changed. The pipeline  `load image → run model → read output`  stayed exactly the same. This is the power of Hugging Face's abstraction.

---
## Section 5  Beyond Images: Text AI

AI isn't just images. Let's try a text model  same pipeline, different modality.

**Sentiment analysis:** Given a sentence, the model predicts whether the sentiment is POSITIVE or NEGATIVE.

In [ ]:
# ── DEMO: TEXT CLASSIFICATION (SENTIMENT ANALYSIS) ───────────────────────────
# DATA: sentences (strings)
# MODEL: DistilBERT fine-tuned on SST-2 (movie review sentiment)
# OUTPUT: label (POSITIVE/NEGATIVE) + confidence score

# The sentiment analyzer was already loaded in Section 0

sentences = [
    "This workshop is actually pretty interesting!",
    "I don't understand any of this, it's overwhelming.",
    "The coffee here is absolutely terrible.",
    "I can't believe how quickly I ran my first AI model.",
    "This is okay I guess, nothing special."
]

print("📊 Sentiment Analysis Results")
print("=" * 60)

for sentence in sentences:
    result = sentiment_analyzer(sentence)[0]
    label = result['label']
    score = result['score']
    icon = "😊" if label == "POSITIVE" else "😞"
    print(f"\n{icon} '{sentence}'")
    print(f"   → {label} ({score:.1%} confidence)")

In [ ]:
# 🟡 YOUR TURN  analyze your own sentences
# Add 3–5 sentences of your own. Try to trick the model!
# ─────────────────────────────────────────────────────────

my_sentences = [
    "Write your first sentence here",
    "Write your second sentence here",
    "Try something sarcastic  does the model understand it?",
]

# ─────────────────────────────────────────────────────────

print("📊 Your Sentiment Results")
print("=" * 60)

for sentence in my_sentences:
    if "Write your" in sentence or "Try something" in sentence:
        print(f"⚠️  Replace placeholder: '{sentence}'")
        continue
    result = sentiment_analyzer(sentence)[0]
    label = result['label']
    score = result['score']
    icon = "😊" if label == "POSITIVE" else "😞"
    print(f"\n{icon} '{sentence}'")
    print(f"   → {label} ({score:.1%} confidence)")

print("\n Did the model get any wrong? What kinds of sentences confuse it?")

---
## Section 6  The AI Task Landscape

So far we've tried image classification and text classification. But there are dozens of AI tasks available on Hugging Face  and they all follow the same pipeline.

Let's run a quick tour.

In [ ]:
# ── DEMO: TEXT GENERATION ────────────────────────────────────────────────────
# A small language model generating text from a prompt

print("Loading text generation model (GPT-2)...")
text_gen = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=60,
    pad_token_id=50256
)

prompt = "Artificial intelligence is changing the world because"

print(f"\n📝 Prompt: '{prompt}'")
print("\n🤖 GPT-2 continues it:")
print("-" * 50)

generated = text_gen(prompt, num_return_sequences=2)
for i, g in enumerate(generated):
    print(f"\nOption {i+1}:")
    print(g['generated_text'])

print("\n⚠️  Note: GPT-2 is from 2019  tiny by today's standards. Tomorrow we run modern LLMs locally.")

In [ ]:
# ── DEMO: ZERO-SHOT CLASSIFICATION ───────────────────────────────────────────
# The model classifies text into categories it was NEVER explicitly trained on
# This is a powerful technique  no training data needed for new categories

print("Loading zero-shot classifier...")
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

text_to_classify = "The government announced new policies to reduce carbon emissions and promote renewable energy."

# You can define ANY categories  no training needed
candidate_labels = ["politics", "environment", "sports", "technology", "entertainment"]

result = zero_shot(text_to_classify, candidate_labels)

print(f"\n📝 Text: '{text_to_classify}'")
print(f"\n🏷️  Categories we asked about: {candidate_labels}")
print("\n📊 Model's confidence in each category:")
print("-" * 45)

for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f"{label:<20} {bar} {score:.1%}")

print("\n💡 Key insight: we defined these categories ourselves.")
print("   The model was never specifically trained to classify news topics.")
print("   This is called 'zero-shot'  works on categories never seen during training.")

In [ ]:
# 🟡 EXPERIMENT: Try zero-shot on your own text and categories
# ─────────────────────────────────────────────────────────────

my_text = "Write any sentence here about any topic"

my_labels = ["category1", "category2", "category3", "category4"]

# ─────────────────────────────────────────────────────────────

if "Write any" not in my_text and "category" not in my_labels[0]:
    result = zero_shot(my_text, my_labels)
    print(f"📝 Text: '{my_text}'")
    print("\n📊 Results:")
    for label, score in zip(result['labels'], result['scores']):
        bar = '█' * int(score * 30)
        print(f"{label:<20} {bar} {score:.1%}")
else:
    print("⚠️  Replace the placeholder text and labels first, then run this cell again.")

---
## Section 7  The Hugging Face Ecosystem

Everything we've used today lives on **Hugging Face**  the largest public repository of AI models.

Let's understand what's available.

In [ ]:
# ── EXPLORE: What tasks does Hugging Face support? ───────────────────────────

task_map = {
    "Natural Language Processing": [
        "Text Classification (sentiment, spam, topic)",
        "Text Generation (GPT-style, story writing)",
        "Question Answering (from a document)",
        "Summarization (condense long text)",
        "Translation (one language → another)",
        "Named Entity Recognition (find names, places, orgs)",
        "Zero-Shot Classification (no training needed)",
    ],
    "Computer Vision": [
        "Image Classification (what object is this?)",
        "Object Detection (where are the objects?)",
        "Image Segmentation (pixel-level labelling)",
        "Depth Estimation (how far away is each pixel?)",
        "Image-to-Text (describe this image)",
    ],
    "Audio": [
        "Automatic Speech Recognition (speech → text)",
        "Audio Classification (what sound is this?)",
        "Text-to-Speech (text → audio)",
    ],
    "Multimodal": [
        "Visual Question Answering (ask questions about images)",
        "Image-Text Matching",
        "Document Understanding",
    ]
}

print("📚 HUGGING FACE TASK LANDSCAPE")
print("=" * 55)
for category, tasks in task_map.items():
    print(f"\n🔷 {category}")
    for task in tasks:
        print(f"   • {task}")

print("\n" + "=" * 55)
print("💡 All of these follow the same pipeline:")
print("   DATA → MODEL → OUTPUT → INTEGRATION")
print("\n🌐 Explore: https://huggingface.co/models")

---
## Section 8  🟡 Go Further (If You're Ahead)

Finished early? Pick any one of these challenges:

**Challenge A  Break the model:**
Find an image that causes the classifier to be confidently *wrong*. What does this tell you about how the model was trained?

**Challenge B  Try a new task:**
Use the `pipeline` function with a task not used today. Try `"image-to-text"` with model `"nlpconnect/vit-gpt2-image-captioning"`. Feed it an image and see if the caption makes sense.

**Challenge C  Benchmark two models:**
Pick the same image, run it through two different image classifiers, compare the outputs. Which is more accurate? Which is faster? (Time them with `%%time` at the top of the cell.)

**Challenge D  Real-world pipeline sketch:**
Without any code, sketch on paper: how would you use today's models to build a system that automatically organises photos in a folder into categories? What are the steps? What are the failure modes?

In [ ]:
# 🟡 MODEL
model_id = "Salesforce/blip-image-captioning-base"

print(f"Loading model: {model_id}\n")

captioner = pipeline(
    "image-text-to-text",
    model=model_id
)

# 🖼️ Image
test_url = "https://cdn.britannica.com/98/235798-050-3C3BA15D/Hamburger-and-french-fries-paper-box.jpg?w=300"
test_image = load_image_from_url(test_url)

# ✅ Display
display(test_image.resize((300, 300)))

# 🧠 Caption
result = captioner(
    images=test_image,
    text="a photo of"
)

print("\n🖼️ Caption:")
print("-" * 50)
print(result[0]['generated_text'])

print("\n🤔 Is this description complete? What details are missing?")

---
## Section 9  Day 1 Recap

Before you close this notebook, make sure you can answer these:

1. **What is the difference between Machine Learning and Deep Learning?**
   > *ML = learning from data. DL = ML using multi-layer neural networks on raw data (images, text, audio).*

2. **What does the Data → Model → Output → Integration pipeline mean?**
   > *Every AI system takes raw input (data), transforms it through a trained network (model), produces a structured result (output), and that result connects to something useful (integration).*

3. **What is Hugging Face and why does it matter?**
   > *A repository of 500,000+ pre-trained models. Instead of training from scratch (months, $$$), you load a model in one line and use it immediately.*

4. **What did you run today?**
   > *At minimum: an image classifier (ViT) and a sentiment analyzer (DistilBERT). Optionally: GPT-2, zero-shot classifier, image captioner.*

---

##  Coming Tomorrow  Day 2: LLMs, Prompting & RAG

Tomorrow you'll:
- Run a **modern LLM locally** on your machine (Ollama + Mistral)
- Learn how to write **effective prompts** that shape model behaviour
- Build a **RAG chatbot** that can answer questions from any document you give it
- Understand *why* LLMs hallucinate and *how* retrieval fixes it

> **Optional prep:** Watch the first 10 minutes of Andrej Karpathy's "Intro to Large Language Models" on YouTube. It'll make tomorrow click much faster.

**See you tomorrow. **